# Etapa 04: Traducción Neuronal MarianMT en Google Colab
### Clasificación y Detección de Spam Enron (EN -> ES) - Grupo 5

Este notebook traslada la carga pesada de inferencia a una GPU acelerada en la nube (**NVIDIA Tesla T4 / A100** de 16 GB VRAM con Tensor Cores),
permitiendo completar la fragmentación de oraciones y traducción neuronal completa en **~1.5 a 2.5 horas** sin consumir recursos locales de tu PC.

---
### Pasos rápidos:
1. Asegúrate de tener asignada una GPU en **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4**.
2. Sube el archivo `colab_pack.zip` (~48 MB generado por tu PC) a la pestaña **Archivos** (a la izquierda).
3. Ejecuta las celdas en orden (o pulsa **Entorno de ejecución > Ejecutar todas**).
4. Al finalizar, la última celda descargará `enron_traducido_es_neural_full.csv` directamente a tu equipo.

## 1. Verificación de GPU en el entorno

In [ ]:
!nvidia-smi

## 2. Instalación de dependencias necesarias

In [ ]:
!pip install -q transformers sentencepiece sacremoses accelerate pandas

## 3. Descomprimir el paquete del proyecto (`colab_pack.zip`)
Si ya subiste `colab_pack.zip` en el explorador de archivos de la izquierda, la siguiente celda lo descomprimirá automáticamente.

In [ ]:
import os, zipfile
from google.colab import files

zip_name = 'colab_pack.zip'
if not os.path.exists(zip_name):
    print(f"[!] No se encontro '{zip_name}' en /content/.")
    print("Por favor, selecciona y sube 'colab_pack.zip' desde tu PC:")
    uploaded = files.upload()

if os.path.exists(zip_name):
    print(f"[+] Descomprimiendo {zip_name}...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("[OK] Archivos del proyecto listos en el entorno:")
    !ls -la src/ data/03_nlp/ data/04_traducido/
else:
    print("[ERROR] No se pudo encontrar colab_pack.zip")

## 4. Ejecución de la Traducción Neuronal con Fragmentación Completa

- **`--patch-existing`**: Conserva los 11,623 correos cortos ya traducidos y procesa únicamente los correos largos (>600 caracteres).
- **`--full-chunks`**: Divide los correos largos en oraciones completas y las traduce sin truncar ningún carácter.
- **`--batch 128`**: Aprovecha los 16 GB de VRAM de la Tesla T4 para máxima velocidad.
- **`--resume`**: Si se interrumpe, retoma automáticamente desde el último checkpoint cada 200 filas.

In [ ]:
!python src/04_traducir_es.py --marian --full-chunks --patch-existing --resume --batch 128 --checkpoint-interval 200

## 5. Inspección y Validación del Dataset Traducido

In [ ]:
import pandas as pd

ruta_salida = 'data/04_traducido/enron_traducido_es_neural_full.csv'
df_res = pd.read_csv(ruta_salida)

print(f"Dimensiones finales: {df_res.shape}")
print(f"Correos con traduccion en espanol: {df_res['texto_es'].notna().sum()}/{len(df_res)}")
print(f"Promedio de fragmentos por correo: {df_res['num_fragmentos'].mean():.2f}")
print(f"Maximo de fragmentos en un correo: {df_res['num_fragmentos'].max()}")

display(df_res[['message_id', 'Category', 'clean_en', 'texto_es', 'num_fragmentos']].head())

## 6. Descargar el archivo final a tu PC
Ejecuta esta celda para descargar `enron_traducido_es_neural_full.csv` directamente a tu carpeta de descargas.
Luego muévelo a tu carpeta local: `data/04_traducido/enron_traducido_es_neural_full.csv`.

In [ ]:
from google.colab import files

print("[+] Iniciando descarga a tu equipo local...")
files.download('data/04_traducido/enron_traducido_es_neural_full.csv')